## 本章为langchain的基础应用，为本人自己总结且可以运行的代码，用于学习langchain

### 1.安装依赖（本文用的包管理工具为uv，若使用pip的同学需可以根据pyproject.toml安装依赖）
langchain：`uv add langchain`

langchain community：`uv add langchain_community`


### 2.apikey管理
这里我们使用到了dotenv来管理apikey，避免在代码中直接暴露apikey

In [ ]:
from platform import system

from langchain_core.messages import SystemMessage

# 创建.env文件，用于存储apikey等信息，这里我配置了Base_url和模型的名称
# LLM 配置
API_KEY=xxxx
BASE_URL=xxxx
MODEL_NAME=xxxx

In [ ]:
# 从.env文件中加载apikey等信息
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
MODEL_NAME = os.getenv("MODEL_NAME")

### 3.创建llm与交互
这里我们使用到了OpenAI来创建llm，也可以使用其他llm，如deepseek等，但是要注意这里的base_url一定要是openai版本的，不能写成anthropic版本的

In [ ]:
from langchain_openai import ChatOpenAI

# 使用OpenAI的客户端创建llm对象
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model_name=MODEL_NAME,
)

# 调用模型
response = llm.invoke("你好")
print(response.content)

In [ ]:
# 这里我们可以看一下response的完整结构
print(response.model_dump())

In [ ]:
# 因此我们可以在调试的时候打印出重要的语句
print(f"""
===== AI Response =====
模型: {response.response_metadata["model_name"]}
输入Token: {response.usage_metadata["input_tokens"]}
输出Token: {response.usage_metadata["output_tokens"]}
推理Token: {response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)}
结束原因: {response.response_metadata["finish_reason"]}

回复内容:
{response.content}
=======================
""")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

"""
我们常见的消息类型一共有四种
1. HumanMessage：用户消息
2. AIMessage：模型回复
3. SystemMessage：系统消息
4. ToolMessage：工具调用消息
由于我们需要与模型交互，所以需要使用HumanMessage来发送用户消息，模型回复则使用AIMessage来接收，而系统消息则使用SystemMessage来设置，工具调用消息则使用ToolMessage来设置模型的工具调用
"""
system_prompt = SystemMessage(content="你是一个专业的翻译，你的任务是将用户输入的中文翻译成英文")
human_prompt = HumanMessage(content="你好，我的名字叫约翰")

llm.invoke([system_prompt, human_prompt])


In [ ]:
# 下面是流式输出，通过每一个thunk进行输出
stream = llm.stream([system_prompt, human_prompt])
for thunk in stream:
    print(thunk.text, end="")


### 4. LangChain Expression Language
其实就是通过 | 来连接上下文，实现链式调用，传递输出结果给下一个组件
其实langchain的本质就是声明组件+连接组件

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 创建一个字符串输出解析器,本质就是获取模型回复的content
parser = StrOutputParser()

# 由于chain的调用需要用到Runnable对象，这里把prompt也转换为Runnable对象
prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    human_prompt,
])

# 构造链
chain = prompt | llm | parser

message = chain.invoke({})
print(message)

Hello, my name is John.
